# PrimoGPT LLM Benchmarking: Vanilla Qwen, QLoRA, and QA-LoRA

This notebook fine-tunes Qwen models using the PrimoGPT training data and runs inference on 5 stocks.

**3 Configurations:**
1. **Vanilla Qwen** - Base model, no fine-tuning, just inference
2. **QLoRA Qwen** - 4-bit quantized base + LoRA adapters (rank r=16)
3. **QA-LoRA Qwen** - 4-bit quantized base + quantization-aware LoRA adapters using PEFT's native QA-LoRA support

**Memory Requirements:**
| Model | 4-bit Inference | QLoRA Training | Recommended GPU |
|-------|-----------------|----------------|-----------------|
| Qwen3.5-27B | ~15 GB | ~25 GB | Colab A100 (40GB) |
| Qwen3.5-9B | ~6 GB | ~12 GB | Colab T4 (16GB) |

**Your Mac (36GB RAM, M4 Max)**: Cannot efficiently fine-tune. Use Colab with A100 GPU runtime.**

## Section 0: Configuration

Change the settings below before running. Select your model and GPU runtime accordingly.

In [ ]:
# ============================================================
# CONFIGURATION - Change these before running
# ============================================================

import os

# Temporary workaround for Hugging Face download outages/timeouts.
# This tells Unsloth to fetch supported models via ModelScope instead.
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Disable wandb integration for this notebook.
# This avoids Colab/runtime issues from a broken wandb import when Unsloth patches training libs.
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Model selection: uncomment ONE model
# Qwen3.5-27B: needs Colab A100 (40GB) for inference, but long-context fine-tuning becomes very memory-heavy.
MODEL_NAME = "unsloth/Qwen3.5-27B"              # ~15GB inference, QLoRA / QA-LoRA can still OOM with aggressive settings
# MODEL_NAME = "Qwen/Qwen3.5-27B"               # Same model, non-Unsloth (slower)

# Fine-tuning hyperparameters
LORA_R = 16                    # LoRA rank
LORA_ALPHA = 16                # LoRA alpha (standard: alpha = r)
QALORA_GROUP_SIZE = 16         # QA-LoRA group size; tune this if you want to study the accuracy/memory trade-off
LEARNING_RATE = 2e-4           # Learning rate
NUM_EPOCHS = 1                 # Number of epochs
BATCH_SIZE = 1                 # Safer for 27B fine-tuning; batch_size=4 at long context can exceed even large GPUs
GRADIENT_ACCUMULATION = 8      # Effective batch size = BATCH_SIZE * GRADIENT_ACCUMULATION = 8
MAX_SEQ_LENGTH = 2048          # Inference / general default
TRAIN_MAX_SEQ_LENGTH = 1024    # QA-LoRA on 27B is much more memory hungry than vanilla inference; start here
WARMUP_STEPS = 5               # Warmup steps (match reference PrimoGPT)
SEED = 3407                    # Random seed (match reference PrimoGPT)

# QLoRA data balancing
QLORA_MAX_ZERO_TO_NONZERO_RATIO = 1.0    # Keep at most this many all-zero labels per non-zero label for QLoRA
QLORA_MIN_ZERO_EXAMPLES = 32             # Preserve some all-zero examples so QLoRA still learns the null case

# Stock tickers for inference
TICKERS = ["AAPL", "AMZN", "CRM", "MSFT", "NFLX"]
START_DATE = "2022-04-01"
END_DATE = "2025-02-28"

# Company profiles (hardcoded to avoid Finnhub API dependency)
COMPANY_PROFILES = {
    "AAPL": {
        "name": "Apple Inc",
        "symbol": "AAPL",
        "exchange": "NASDAQ NMS - GLOBAL MARKET",
        "industry": "Technology",
        "marketCapitalization": 3287742,  # in millions
        "employeeTotal": 161000
    },
    "AMZN": {
        "name": "Amazon.com Inc",
        "symbol": "AMZN",
        "exchange": "NASDAQ NMS - GLOBAL MARKET",
        "industry": "Retail Trade",
        "marketCapitalization": 1922890,
        "employeeTotal": 1525000
    },
    "CRM": {
        "name": "Salesforce Inc",
        "symbol": "CRM",
        "exchange": "NEW YORK STOCK EXCHANGE",
        "industry": "Technology",
        "marketCapitalization": 286400,
        "employeeTotal": 79390
    },
    "MSFT": {
        "name": "Microsoft Corp",
        "symbol": "MSFT",
        "exchange": "NASDAQ NMS - GLOBAL MARKET",
        "industry": "Technology",
        "marketCapitalization": 3132600,
        "employeeTotal": 228000
    },
    "NFLX": {
        "name": "Netflix Inc",
        "symbol": "NFLX",
        "exchange": "NASDAQ NMS - GLOBAL MARKET",
        "industry": "Communications",
        "marketCapitalization": 310700,
        "employeeTotal": 13000
    }
}

# Output directories
OUTPUT_DIR = "/content/drive/MyDrive/primogpt_output"
DATA_DIR = "/content/drive/MyDrive/primogpt_data"

print(f"Model: {MODEL_NAME}")
print(f"LoRA rank: {LORA_R}, Alpha: {LORA_ALPHA}")
print(f"QA-LoRA group size: {QALORA_GROUP_SIZE}")
print(f"Per-device batch size: {BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"Inference/default max sequence length: {MAX_SEQ_LENGTH}")
print(f"Training max sequence length: {TRAIN_MAX_SEQ_LENGTH}")
print(f"QLoRA zero/non-zero cap: {QLORA_MAX_ZERO_TO_NONZERO_RATIO}:1 (min zero examples: {QLORA_MIN_ZERO_EXAMPLES})")
print(f"Stocks: {TICKERS}")
print(f"UNSLOTH_USE_MODELSCOPE={os.environ['UNSLOTH_USE_MODELSCOPE']}")
print(f"PYTORCH_CUDA_ALLOC_CONF={os.environ['PYTORCH_CUDA_ALLOC_CONF']}")
print(f"WANDB_DISABLED={os.environ['WANDB_DISABLED']}")
print("For 27B QA-LoRA fine-tuning, increase training sequence length only after confirming stable memory headroom.")

## Section 1: Install Dependencies

In [ ]:
# Install Unsloth and dependencies
%pip install modelscope
%pip install --upgrade --no-deps "peft>=0.18.0" trl accelerate bitsandbytes
%pip install --upgrade wandb
%pip install --no-deps xformers==0.0.29.post1
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --no-deps unsloth_zoo
%pip install datasets transformers huggingface_hub

# For data processing
%pip install pandas numpy yfinance finnhub-python

print("Dependency install complete. If this is a fresh Colab runtime, run the next bootstrap cell before any training cells.")

In [ ]:
# Bootstrap training libraries in the safest import order for Colab.
# Run this once after installing packages and before any QLoRA / QA-LoRA cells.

import os
import sys

os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# If these libraries were already imported in this runtime, Unsloth may not patch them correctly.
already_loaded = [name for name in ("transformers", "trl", "peft", "wandb") if name in sys.modules]
if already_loaded:
    print("Restart the Colab runtime, then run cells from the top through this bootstrap cell.")
    print(f"Already imported before Unsloth bootstrap: {already_loaded}")
else:
    import unsloth
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from trl import SFTTrainer
    from transformers import TrainingArguments
    from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
    print("Bootstrap complete: Unsloth imported before trl / transformers / peft.")
    print(f"UNSLOTH_USE_MODELSCOPE={os.environ['UNSLOTH_USE_MODELSCOPE']}")
    print(f"WANDB_DISABLED={os.environ['WANDB_DISABLED']}")

## Section 2: Mount Google Drive & Load Data

Upload these files to your Google Drive `primogpt_data` folder:
- `merged_data.json` (training data, from `PrimoGPT-main/notebooks/2. Train PrimoGPT model/data/`)
- `AAPL_2022-04-01_2025-02-28.csv` (raw stock data with News column)
- `AMZN_2022-04-01_2025-02-28.csv`
- `CRM_2022-04-01_2025-02-28.csv`
- `MSFT_2022-04-01_2025-02-28.csv`
- `NFLX_2022-04-01_2025-02-28.csv`

These files are in `PrimoGPT-main/notebooks/5. Generate NLP features with PrimoGPT/downloaded_data/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Verify data files exist
training_data_path = os.path.join(DATA_DIR, "merged_data.json")
stock_data_paths = {
    ticker: os.path.join(DATA_DIR, f"{ticker}_{START_DATE}_{END_DATE}.csv")
    for ticker in TICKERS
}

print("Checking data files...")
if os.path.exists(training_data_path):
    import json
    with open(training_data_path) as f:
        training_data = json.load(f)
    print(f"  Training data: {len(training_data)} records")
else:
    print(f"  ERROR: Training data not found at {training_data_path}")
    print(f"  Please upload merged_data.json to {DATA_DIR}")

for ticker, path in stock_data_paths.items():
    if os.path.exists(path):
        import pandas as pd
        df = pd.read_csv(path)
        print(f"  {ticker}: {len(df)} rows, columns={list(df.columns)[:6]}...")
    else:
        print(f"  WARNING: {ticker} data not found at {path}")
        print(f"  Please upload {ticker}_{START_DATE}_{END_DATE}.csv to {DATA_DIR}")

In [ ]:
import json
import pandas as pd
import numpy as np
import re
import torch
from tqdm import tqdm

# Load training data
with open(training_data_path) as f:
    training_data = json.load(f)

print(f"Loaded {len(training_data)} training records")
print(f"Sample instruction: {training_data[0]['instruction'][:100]}...")
print(f"Sample response: {training_data[0]['response']}")

## Section 3: Vanilla Qwen Inference (No Fine-Tuning)

Load the base Qwen model and run inference on all 5 stocks. This serves as the baseline.

In [ ]:
import os

# Re-apply the fallback here so this section also works if run independently.
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"

from unsloth import FastLanguageModel

print("Loading base Qwen model (Vanilla) via Unsloth with ModelScope fallback...")
model_vanilla, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect
    load_in_4bit=True,  # Use 4-bit quantization to save memory
)
FastLanguageModel.for_inference(model_vanilla)  # Enable inference mode

# Print model info
total_params = sum(p.numel() for p in model_vanilla.parameters())
trainable_params = sum(p.numel() for p in model_vanilla.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"UNSLOTH_USE_MODELSCOPE={os.environ['UNSLOTH_USE_MODELSCOPE']}")

In [ ]:
# ============================================================
# Prompt formatting and inference functions
# ============================================================

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

SYSTEM_PROMPT = """[SYSTEM PROMPT] You are a senior quantitative analyst specializing in stock market analysis. Your task is to analyze the provided company recent news and press releases to generate key features that could influence the stock's price movement in the next trading session. Focus on interpreting the given data to provide insights for algorithmic trading models."""

INPUT_TEMPLATE = """[COMPANY BASICS]
{company_info}

[RECENT NEWS]
Here are the recent news articles related to {symbol}:
{news}

[LATEST PRESS RELEASE]
Here is the most recent press release from {symbol}:
{press_releases}

[ANALYSIS TASKS]
Based primarily on the provided news and press releases (if available) generate the following features defined in the output format. When analyzing, pay special attention to:

1. Potential negative impacts or risks, even if they're subtle or not the main focus of the news.
2. Market saturation signs, increased competition, or regulatory challenges.
3. Discrepancies between the tone of the news and the actual content.
4. Possible overoptimism in positive news that might lead to unrealistic expectations.
5. Short-term versus long-term implications of the news, especially potential short-term negative reactions.

Be cautious of overly positive sentiment and ensure you're giving appropriate weight to any negative indicators.

[OUTPUT FORMAT]
The output should be a JSON object with the following fields:
- news_relevance: How directly relevant the news is to stock performance (0: not relevant, 1: somewhat relevant, 2: highly relevant)
- sentiment: Overall sentiment (-1: negative, 0: neutral, 1: positive)
- price_impact_potential: Potential price impact (-3: strong negative to 3: strong positive)
- trend_direction: Likely direction (-1: downward, 0: neutral, 1: upward)
- earnings_impact: Impact on future earnings (-2: significant negative to 2: significant positive)
- investor_confidence: Effect on investor confidence (-3: major decrease to 3: major increase)
- risk_profile_change: Change in risk profile (-2: significantly increased risk to 2: significantly decreased risk)

IMPORTANT: Your response should ONLY include the JSON structure. Do not include any additional explanation."""

def format_company_info(profile, adj_close, bin_label):
    """Format company information for the prompt."""
    price_change_desc = bin_label.replace('U', 'up by ').replace('D', 'down by ')
    price_change_desc = price_change_desc.replace('1', '0-1%').replace('2', '1-2%').replace('3', '2-3%').replace('4', '3-4%')
    if '5+' in price_change_desc:
        price_change_desc = price_change_desc.replace('5+', 'more than 5%')
    else:
        price_change_desc = price_change_desc.replace('5', '4-5%')
    
    return (f"{profile['name']} is a company trading under the ticker {profile['symbol']}. "
            f"The company operates in the {profile['industry']} industry with a market capitalization of "
            f"${profile['marketCapitalization']:,.0f}. "
            f"It has {profile['employeeTotal']:,} employees. "
            f"The current stock price is ${adj_close:.2f}, with today's price change {price_change_desc} compared to the previous closing price.")

def format_news(news_json_str):
    """Format news from JSON string."""
    try:
        news = json.loads(news_json_str)
        if not news:
            return "No recent news available."
        # Sample up to 10 news items
        sampled = news[:10]
        return "\n".join([f"[Headline]: {n['headline']}\n[Summary]: {n['summary']}" for n in sampled])
    except:
        return "No recent news available."

def format_press_releases(pr_json_str):
    """Format press releases from JSON string."""
    try:
        prs = json.loads(pr_json_str)
        if not prs:
            return "No recent press releases available."
        return "\n".join([f"[Headline]: {pr['headline']}\n[Description]: {pr['description']}" for pr in prs[:5]])
    except:
        return "No recent press releases available."

def parse_model_response(response_text):
    """Parse the model's JSON response into feature dict."""
    # Extract response from Alpaca format
    response_match = re.search(r'### Response:\s*(.*)', response_text, re.DOTALL)
    if response_match:
        response_text = response_match.group(1).strip()
    
    # Try to extract JSON
    json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
    if json_match:
        try:
            result = json.loads(json_match.group(0))
            # Normalize keys
            normalized = {}
            for key, expected in [('news_relevance', 'News Relevance'), ('sentiment', 'Sentiment'),
                                   ('price_impact_potential', 'Price Impact Potential'), 
                                   ('trend_direction', 'Trend Direction'),
                                   ('earnings_impact', 'Earnings Impact'),
                                   ('investor_confidence', 'Investor Confidence'),
                                   ('risk_profile_change', 'Risk Profile Change')]:
                val = result.get(key, 0)
                try:
                    normalized[expected] = int(val)
                except (ValueError, TypeError):
                    normalized[expected] = 0
            return normalized
        except json.JSONDecodeError:
            pass
    
    # Fallback: return zeros
    return {
        'News Relevance': 0, 'Sentiment': 0, 'Price Impact Potential': 0,
        'Trend Direction': 0, 'Earnings Impact': 0, 
        'Investor Confidence': 0, 'Risk Profile Change': 0
    }

print("Helper functions defined.")

In [ ]:
# ============================================================
# Run Vanilla Qwen Inference on All 5 Stocks
# ============================================================

def run_inference(model, tokenizer, stock_data_path, ticker, model_label="vanilla"):
    """Run inference on a single stock and return results DataFrame."""
    df = pd.read_csv(stock_data_path)
    profile = COMPANY_PROFILES[ticker]
    results = []
    
    for i in tqdm(range(len(df)), desc=f"{ticker} ({model_label})"):
        row = df.iloc[i]
        
        # Format company info
        adj_close = row['Adj Close Price']
        bin_label = row['Bin Label']
        company_info = format_company_info(profile, adj_close, bin_label)
        
        # Format news and press releases
        news_str = format_news(row['News']) if pd.notna(row['News']) else "No recent news available."
        pr_str = format_press_releases(row['PressReleases']) if pd.notna(row['PressReleases']) else "No recent press releases available."
        
        # Build input
        input_text = INPUT_TEMPLATE.format(
            company_info=company_info,
            symbol=ticker,
            news=news_str,
            press_releases=pr_str
        )
        
        # Build prompt
        prompt = ALPACA_PROMPT.format(SYSTEM_PROMPT, input_text, "")
        
        # Tokenize as text explicitly; Qwen processors may treat a positional arg as an image input
        inputs = tokenizer(text=[prompt], return_tensors="pt")
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        prompt_length = inputs["input_ids"].shape[1]
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,  # JSON response is short; smaller cap is faster and safer
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
                # No temperature/do_sample = greedy decoding (match reference)
            )
        
        generated_tokens = outputs[:, prompt_length:]
        response = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
        features = parse_model_response(response)
        
        results.append({
            'Date': row['Date'],
            'Adj Close Price': row['Adj Close Price'],
            'Returns': row['Returns'],
            'Bin Label': row['Bin Label'],
            'News Relevance': features['News Relevance'],
            'Sentiment': features['Sentiment'],
            'Price Impact Potential': features['Price Impact Potential'],
            'Trend Direction': features['Trend Direction'],
            'Earnings Impact': features['Earnings Impact'],
            'Investor Confidence': features['Investor Confidence'],
            'Risk Profile Change': features['Risk Profile Change'],
            'Prompt': input_text
        })
    
    results_df = pd.DataFrame(results)
    
    # Save to CSV
    output_path = os.path.join(OUTPUT_DIR, f"qwen_{model_label}_ppo_{ticker}_data.csv")
    results_df.to_csv(output_path, index=False)
    print(f"Saved {len(results_df)} rows to {output_path}")
    
    # Print stats
    nlp_cols = ['News Relevance', 'Sentiment', 'Price Impact Potential', 
                'Trend Direction', 'Earnings Impact', 'Investor Confidence', 'Risk Profile Change']
    all_zero = (results_df[nlp_cols] == 0).all(axis=1).sum()
    non_zero = len(results_df) - all_zero
    print(f"  Non-zero predictions: {non_zero}/{len(results_df)} ({non_zero/len(results_df)*100:.1f}%)")
    print(f"  Feature means: {results_df[nlp_cols].mean().to_dict()}")
    
    return results_df

# Run vanilla inference on all 5 stocks
vanilla_results = {}
for ticker in TICKERS:
    path = stock_data_paths[ticker]
    if os.path.exists(path):
        print(f"\n{'='*60}")
        print(f"Running Vanilla Qwen inference on {ticker}...")
        print(f"{'='*60}")
        vanilla_results[ticker] = run_inference(model_vanilla, tokenizer, path, ticker, "base")
    else:
        print(f"Skipping {ticker}: data file not found")

# Free vanilla model memory
del model_vanilla
torch.cuda.empty_cache()
print("\nVanilla inference complete. Model unloaded from GPU.")

## Section 4: QLoRA Fine-Tuning

Fine-tune the Qwen model using QLoRA (4-bit quantized base + LoRA adapters).

**QLoRA** uses standard LoRA with rank r=16, targeting all linear projection layers.

In [ ]:
# ============================================================
# Prepare Training Dataset
# ============================================================

import ast
import json

from datasets import Dataset
from transformers import AutoTokenizer

if "tokenizer" not in globals() or tokenizer is None:
    print("Tokenizer not found in session; loading tokenizer for dataset formatting...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

EOS_TOKEN = tokenizer.eos_token or ""
SIGNAL_KEYS = [
    "news_relevance",
    "sentiment",
    "price_impact_potential",
    "trend_direction",
    "earnings_impact",
    "investor_confidence",
    "risk_profile_change",
]

def parse_training_response(raw_response):
    parsed = raw_response
    if isinstance(raw_response, str):
        try:
            parsed = json.loads(raw_response)
        except json.JSONDecodeError:
            try:
                parsed = ast.literal_eval(raw_response)
            except (ValueError, SyntaxError):
                parsed = None

    if not isinstance(parsed, dict):
        return None, "response is not a JSON object"

    normalized = {}
    for key in SIGNAL_KEYS:
        if key not in parsed:
            return None, f"missing key: {key}"
        value = parsed[key]
        if value is None:
            return None, f"null value: {key}"
        try:
            normalized[key] = int(str(value).strip())
        except (ValueError, TypeError):
            return None, f"non-integer value for {key}: {value!r}"

    return normalized, ""

def build_clean_training_dataset(records, tokenizer, alpaca_prompt, verbose=True):
    eos_token = tokenizer.eos_token or ""
    clean_rows = []
    invalid_reason_counts = {}
    invalid_examples = []
    zero_signal_count = 0

    for index, item in enumerate(records):
        instruction = item.get("instruction")
        input_text = item.get("input")
        if not isinstance(instruction, str) or not isinstance(input_text, str):
            reason = "missing instruction/input text"
            invalid_reason_counts[reason] = invalid_reason_counts.get(reason, 0) + 1
            if len(invalid_examples) < 5:
                invalid_examples.append({"index": index, "reason": reason})
            continue

        normalized_response, reason = parse_training_response(item.get("response"))
        if normalized_response is None:
            invalid_reason_counts[reason] = invalid_reason_counts.get(reason, 0) + 1
            if len(invalid_examples) < 5:
                invalid_examples.append({"index": index, "reason": reason})
            continue

        zero_signal_count += int(all(normalized_response[key] == 0 for key in SIGNAL_KEYS))
        clean_rows.append({
            "instruction": instruction,
            "input": input_text,
            "output": json.dumps(normalized_response, ensure_ascii=False),
        })

    if not clean_rows:
        raise ValueError("No valid training rows remained after filtering invalid labels.")

    dataset_dict = {
        "instruction": [row["instruction"] for row in clean_rows],
        "input": [row["input"] for row in clean_rows],
        "output": [row["output"] for row in clean_rows],
    }
    dataset = Dataset.from_dict(dataset_dict)

    def formatting_prompts_func(examples):
        texts = []
        for instruction, input_text, output in zip(examples["instruction"], examples["input"], examples["output"]):
            texts.append(alpaca_prompt.format(instruction, input_text, output) + eos_token)
        return {"text": texts}

    dataset = dataset.map(formatting_prompts_func, batched=True)

    stats = {
        "raw_rows": len(records),
        "valid_rows": len(clean_rows),
        "dropped_rows": len(records) - len(clean_rows),
        "zero_signal_rows": zero_signal_count,
        "non_zero_signal_rows": len(clean_rows) - zero_signal_count,
        "invalid_reason_counts": invalid_reason_counts,
        "invalid_examples": invalid_examples,
    }

    if verbose:
        print(f"Loaded {stats['raw_rows']} raw training rows")
        print(f"Kept {stats['valid_rows']} valid rows and dropped {stats['dropped_rows']} invalid rows")
        print(f"All-zero label rows kept: {stats['zero_signal_rows']}")
        print(f"Non-zero label rows kept: {stats['non_zero_signal_rows']}")
        if invalid_reason_counts:
            print("Dropped-row reasons:")
            for reason, count in sorted(invalid_reason_counts.items(), key=lambda item: (-item[1], item[0])):
                print(f"  - {reason}: {count}")
        if invalid_examples:
            print("Sample dropped rows:")
            for sample in invalid_examples:
                print(f"  - index {sample['index']}: {sample['reason']}")

    return dataset, clean_rows, stats

dataset, clean_training_records, training_dataset_stats = build_clean_training_dataset(
    training_data,
    tokenizer=tokenizer,
    alpaca_prompt=ALPACA_PROMPT,
    verbose=True,
)

print(f"Training dataset: {len(dataset)} examples")
print(f"Sample formatted text (first 500 chars):\n{dataset[0]['text'][:500]}")

In [ ]:
# ============================================================
# Load model for QLoRA fine-tuning
# ============================================================

import os

os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

if "FastLanguageModel" not in globals():
    raise RuntimeError(
        "Run the bootstrap cell near the top of the notebook first so Unsloth loads before trl / transformers / peft."
    )

print("Loading model for QLoRA fine-tuning...")
model_qlora, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=TRAIN_MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Add QLoRA adapters
# Standard QLoRA: rank r=16, alpha=16 (1:1 ratio), targeting all projection layers
model_qlora = FastLanguageModel.get_peft_model(
    model_qlora,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

if hasattr(model_qlora, "config"):
    model_qlora.config.use_cache = False

# Print trainable parameters
trainable_params = sum(p.numel() for p in model_qlora.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_qlora.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")
print(f"Training max sequence length: {TRAIN_MAX_SEQ_LENGTH}")
print(f"UNSLOTH_USE_MODELSCOPE={os.environ['UNSLOTH_USE_MODELSCOPE']}")
print(f"WANDB_DISABLED={os.environ['WANDB_DISABLED']}")

In [ ]:
# ============================================================
# Train QLoRA model
# ============================================================

if "SFTTrainer" not in globals() or "TrainingArguments" not in globals() or "is_bfloat16_supported" not in globals():
    raise RuntimeError(
        "Run the bootstrap cell near the top of the notebook first so the training libraries are initialized safely."
    )

if "dataset" not in globals() or dataset is None:
    print("Training dataset not found in memory; rebuilding the cleaned dataset for this runtime...")
    import ast
    import json
    from datasets import Dataset
    from transformers import AutoTokenizer

    if "training_data" not in globals() or training_data is None:
        if "training_data_path" not in globals():
            raise RuntimeError("Run the Google Drive/data loading cells first so training_data_path is defined.")
        with open(training_data_path) as file:
            training_data = json.load(file)

    if "ALPACA_PROMPT" not in globals():
        ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

    if "tokenizer" not in globals() or tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
        if tokenizer.pad_token is None and tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token

    if "SIGNAL_KEYS" not in globals():
        SIGNAL_KEYS = [
            "news_relevance",
            "sentiment",
            "price_impact_potential",
            "trend_direction",
            "earnings_impact",
            "investor_confidence",
            "risk_profile_change",
        ]

    if "parse_training_response" not in globals():
        def parse_training_response(raw_response):
            parsed = raw_response
            if isinstance(raw_response, str):
                try:
                    parsed = json.loads(raw_response)
                except json.JSONDecodeError:
                    try:
                        parsed = ast.literal_eval(raw_response)
                    except (ValueError, SyntaxError):
                        parsed = None

            if not isinstance(parsed, dict):
                return None, "response is not a JSON object"

            normalized = {}
            for key in SIGNAL_KEYS:
                if key not in parsed:
                    return None, f"missing key: {key}"
                value = parsed[key]
                if value is None:
                    return None, f"null value: {key}"
                try:
                    normalized[key] = int(str(value).strip())
                except (ValueError, TypeError):
                    return None, f"non-integer value for {key}: {value!r}"

            return normalized, ""

    if "build_clean_training_dataset" not in globals():
        def build_clean_training_dataset(records, tokenizer, alpaca_prompt, verbose=True):
            eos_token = tokenizer.eos_token or ""
            clean_rows = []
            invalid_reason_counts = {}
            invalid_examples = []
            zero_signal_count = 0

            for index, item in enumerate(records):
                instruction = item.get("instruction")
                input_text = item.get("input")
                if not isinstance(instruction, str) or not isinstance(input_text, str):
                    reason = "missing instruction/input text"
                    invalid_reason_counts[reason] = invalid_reason_counts.get(reason, 0) + 1
                    if len(invalid_examples) < 5:
                        invalid_examples.append({"index": index, "reason": reason})
                    continue

                normalized_response, reason = parse_training_response(item.get("response"))
                if normalized_response is None:
                    invalid_reason_counts[reason] = invalid_reason_counts.get(reason, 0) + 1
                    if len(invalid_examples) < 5:
                        invalid_examples.append({"index": index, "reason": reason})
                    continue

                zero_signal_count += int(all(normalized_response[key] == 0 for key in SIGNAL_KEYS))
                clean_rows.append({
                    "instruction": instruction,
                    "input": input_text,
                    "output": json.dumps(normalized_response, ensure_ascii=False),
                })

            if not clean_rows:
                raise ValueError("No valid training rows remained after filtering invalid labels.")

            dataset_dict = {
                "instruction": [row["instruction"] for row in clean_rows],
                "input": [row["input"] for row in clean_rows],
                "output": [row["output"] for row in clean_rows],
            }
            dataset = Dataset.from_dict(dataset_dict)

            def formatting_prompts_func(examples):
                texts = []
                for instruction, input_text, output in zip(examples["instruction"], examples["input"], examples["output"]):
                    texts.append(alpaca_prompt.format(instruction, input_text, output) + eos_token)
                return {"text": texts}

            dataset = dataset.map(formatting_prompts_func, batched=True)

            stats = {
                "raw_rows": len(records),
                "valid_rows": len(clean_rows),
                "dropped_rows": len(records) - len(clean_rows),
                "zero_signal_rows": zero_signal_count,
                "non_zero_signal_rows": len(clean_rows) - zero_signal_count,
                "invalid_reason_counts": invalid_reason_counts,
                "invalid_examples": invalid_examples,
            }

            if verbose:
                print(f"Loaded {stats['raw_rows']} raw training rows")
                print(f"Kept {stats['valid_rows']} valid rows and dropped {stats['dropped_rows']} invalid rows")
                print(f"All-zero label rows kept: {stats['zero_signal_rows']}")
                print(f"Non-zero label rows kept: {stats['non_zero_signal_rows']}")
                if invalid_reason_counts:
                    print("Dropped-row reasons:")
                    for reason, count in sorted(invalid_reason_counts.items(), key=lambda item: (-item[1], item[0])):
                        print(f"  - {reason}: {count}")
                if invalid_examples:
                    print("Sample dropped rows:")
                    for sample in invalid_examples:
                        print(f"  - index {sample['index']}: {sample['reason']}")

            return dataset, clean_rows, stats

    dataset, clean_training_records, training_dataset_stats = build_clean_training_dataset(
        training_data,
        tokenizer=tokenizer,
        alpaca_prompt=ALPACA_PROMPT,
        verbose=True,
    )
    print(f"Rebuilt cleaned training dataset with {len(dataset)} examples")

gpu_mem_before = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory before training: {gpu_mem_before:.2f} GB")
if gpu_mem_before > 60:
    print("WARNING: GPU memory is already extremely high before training.")
    print("Recommended safe starting point: BATCH_SIZE=1 and TRAIN_MAX_SEQ_LENGTH=1024 in Cell 3, then restart the runtime.")

print(f"Starting QLoRA training ({NUM_EPOCHS} epoch, {len(dataset)} samples)...")
print(f"Training config: batch_size={BATCH_SIZE}, grad_accum={GRADIENT_ACCUMULATION}, max_seq_length={TRAIN_MAX_SEQ_LENGTH}")
if "training_dataset_stats" in globals():
    print(f"Training rows kept after label validation: {training_dataset_stats['valid_rows']}")
    print(f"Training rows dropped after label validation: {training_dataset_stats['dropped_rows']}")

if hasattr(model_qlora, "config"):
    model_qlora.config.use_cache = False

torch.cuda.empty_cache()

qlora_trainer = SFTTrainer(
    model=model_qlora,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=TRAIN_MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        gradient_checkpointing=True,
        logging_steps=10,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir=os.path.join(OUTPUT_DIR, "qlora_checkpoints"),
        report_to="none",
    ),
)

qlora_stats = qlora_trainer.train()

print(f"\nTraining complete!")
print(f"  Training time: {qlora_stats.metrics['train_runtime']:.1f} seconds")
print(f"  Training loss: {qlora_stats.metrics['train_loss']:.4f}")
print(f"  GPU memory after training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ============================================================
# Save QLoRA model
# ============================================================

import shutil

qlora_save_path = os.path.join(OUTPUT_DIR, "qwen_qlora_adapter")
model_qlora.save_pretrained(qlora_save_path)
tokenizer.save_pretrained(qlora_save_path)
print(f"QLoRA adapter saved to {qlora_save_path}")

# Also try to save a merged model for easier inference, but keep the adapter as the canonical artifact
qlora_merged_path = os.path.join(OUTPUT_DIR, "qwen_qlora_merged")
if os.path.exists(qlora_merged_path):
    shutil.rmtree(qlora_merged_path, ignore_errors=True)

try:
    model_qlora.save_pretrained_merged(qlora_merged_path, tokenizer, save_method="merged_16bit")
    print(f"QLoRA merged model saved to {qlora_merged_path}")
except Exception as error:
    print(f"Merged QLoRA export failed: {type(error).__name__}: {error}")
    if os.path.exists(qlora_merged_path):
        shutil.rmtree(qlora_merged_path, ignore_errors=True)
        print("Removed incomplete merged QLoRA directory after failed export.")
    print("Continue with the saved adapter path; the QLoRA inference cell will reload the base model + adapter.")

In [ ]:
# ============================================================
# Run QLoRA Inference on All 5 Stocks
# ============================================================

import os
import json
import ast
import re
import glob
import shutil
import pandas as pd
import torch
from tqdm import tqdm

if "FastLanguageModel" not in globals():
    raise RuntimeError("Run the bootstrap cell first so `FastLanguageModel` is available.")

from peft import PeftModel

qlora_merged_path = os.path.join(OUTPUT_DIR, "qwen_qlora_merged")
qlora_adapter_path = os.path.join(OUTPUT_DIR, "qwen_qlora_adapter")

def has_valid_merged_checkpoint(model_dir):
    if not os.path.isdir(model_dir):
        return False
    if os.path.isfile(os.path.join(model_dir, "adapter_config.json")):
        return False
    if not os.path.isfile(os.path.join(model_dir, "config.json")):
        return False
    candidate_patterns = ["model*.safetensors", "*.safetensors", "*.bin"]
    for pattern in candidate_patterns:
        for file_path in glob.glob(os.path.join(model_dir, pattern)):
            file_name = os.path.basename(file_path).lower()
            if "adapter" in file_name:
                continue
            try:
                if os.path.getsize(file_path) > 0:
                    return True
            except OSError:
                continue
    return False

merged_available = has_valid_merged_checkpoint(qlora_merged_path)
adapter_available = os.path.isdir(qlora_adapter_path)

if not merged_available and os.path.isdir(qlora_merged_path):
    print(f"Skipping invalid merged QLoRA directory (missing full-model weights or adapter-only artifacts): {qlora_merged_path}")

if not merged_available and not adapter_available:
    raise RuntimeError(
        "No valid saved `qwen_qlora_merged` weights or `qwen_qlora_adapter` folder found in OUTPUT_DIR. "
        "Run the QLoRA save cell first so the model is written to Google Drive."
    )

model_qlora = None
last_load_error = None

if merged_available:
    try:
        print(f"Trying merged QLoRA model from Drive: {qlora_merged_path}")
        merged_dtype = torch.bfloat16 if ("is_bfloat16_supported" in globals() and is_bfloat16_supported()) else torch.float16
        model_qlora, tokenizer = FastLanguageModel.from_pretrained(
            model_name=qlora_merged_path,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=merged_dtype,
            load_in_4bit=False,
        )
        print("Loaded merged QLoRA model successfully.")
    except Exception as error:
        last_load_error = error
        print(f"Merged QLoRA load failed: {type(error).__name__}: {error}")
        if os.path.isdir(qlora_merged_path):
            shutil.rmtree(qlora_merged_path, ignore_errors=True)
            print("Removed corrupted merged QLoRA directory after failed load.")
        print("Falling back to base model + saved QLoRA adapter.")

if model_qlora is None:
    if not adapter_available:
        raise RuntimeError(
            "Merged QLoRA checkpoint could not be loaded, and `qwen_qlora_adapter` was not found in OUTPUT_DIR."
        ) from last_load_error

    print(f"Loading base model for adapter inference: {MODEL_NAME}")
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    print(f"Loading QLoRA adapter from Drive: {qlora_adapter_path}")
    model_qlora = PeftModel.from_pretrained(base_model, qlora_adapter_path)
    print("Loaded QLoRA adapter on top of the base model successfully.")

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

SYSTEM_PROMPT = """[SYSTEM PROMPT] You are a senior quantitative analyst specializing in stock market analysis. Your task is to analyze the provided company recent news and press releases to generate key features that could influence the stock's price movement in the next trading session. Focus on interpreting the given data to provide insights for algorithmic trading models."""

INPUT_TEMPLATE = """[COMPANY BASICS]
{company_info}

[RECENT NEWS]
Here are the recent news articles related to {symbol}:
{news}

[LATEST PRESS RELEASE]
Here is the most recent press release from {symbol} (if available):
{press_releases}

[ANALYSIS TASKS]
Based primarily on the provided news and press releases (if available) generate the following features defined in the output format. When analyzing, pay special attention to:
1. Potential negative impacts or risks, even if they're subtle or not the main focus of the news.
2. Market saturation signs, increased competition, or regulatory challenges.
3. Discrepancies between the tone of the news and the actual content.
4. Possible overoptimism in positive news that might lead to unrealistic expectations.
5. Short-term versus long-term implications of the news, especially potential short-term negative reactions.
Be cautious of overly positive sentiment and ensure you're giving appropriate weight to any negative indicators.

[FUTURE INFORMATION]
Future price information is unavailable at inference time. Generate the features using only the company basics, recent news, and press releases.

[OUTPUT FORMAT]
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing ```json and ```:
```json
{{
  "news_relevance": string,
  "sentiment": string,
  "price_impact_potential": string,
  "trend_direction": string,
  "earnings_impact": string,
  "investor_confidence": string,
  "risk_profile_change": string
}}
```
IMPORTANT: Your response should ONLY include the JSON structure or the markdown code snippet containing it. Do not include any additional explanation or analysis."""

def format_company_info(profile, adj_close, bin_label):
    price_change_desc = bin_label.replace('U', 'up by ').replace('D', 'down by ')
    price_change_desc = price_change_desc.replace('1', '0-1%').replace('2', '1-2%').replace('3', '2-3%').replace('4', '3-4%')
    if '5+' in price_change_desc:
        price_change_desc = price_change_desc.replace('5+', 'more than 5%')
    else:
        price_change_desc = price_change_desc.replace('5', '4-5%')
    return (
        f"{profile['name']} is a company trading under the ticker {profile['symbol']}. "
        f"The company operates in the {profile['industry']} industry with a market capitalization of "
        f"${profile['marketCapitalization']:,.0f}. "
        f"It has {profile['employeeTotal']:,} employees. "
        f"The current stock price is ${adj_close:.2f}, with today's price change {price_change_desc} compared to the previous closing price."
    )

def format_news(news_json_str):
    try:
        news = json.loads(news_json_str)
        if not news:
            return "No recent news available."
        sampled = news[:10]
        return "\n".join([f"[Headline]: {item['headline']}\n[Summary]: {item['summary']}" for item in sampled])
    except Exception:
        return "No recent news available."

def format_press_releases(pr_json_str):
    try:
        press_releases = json.loads(pr_json_str)
        if not press_releases:
            return "No recent press releases available."
        return "\n".join([f"[Headline]: {item['headline']}\n[Description]: {item['description']}" for item in press_releases[:5]])
    except Exception:
        return "No recent press releases available."

def _normalize_key(key):
    return re.sub(r'[^a-z0-9]+', '', str(key).strip().lower())

def empty_feature_dict(fill_value=pd.NA):
    return {
        'News Relevance': fill_value,
        'Sentiment': fill_value,
        'Price Impact Potential': fill_value,
        'Trend Direction': fill_value,
        'Earnings Impact': fill_value,
        'Investor Confidence': fill_value,
        'Risk Profile Change': fill_value,
    }

def parse_model_response(response_text):
    cleaned = response_text.strip()
    response_match = re.search(r'### Response:\s*(.*)', cleaned, re.DOTALL)
    if response_match:
        cleaned = response_match.group(1).strip()

    cleaned = re.sub(r'^```json\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'^```\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)

    json_match = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if not json_match:
        return {
            'features': empty_feature_dict(),
            'parse_success': False,
            'parse_error': 'No JSON object found in model response.',
        }

    candidate = json_match.group(0)
    parsed = None
    try:
        parsed = json.loads(candidate)
    except json.JSONDecodeError:
        try:
            parsed = ast.literal_eval(candidate)
        except (ValueError, SyntaxError):
            parsed = None

    if not isinstance(parsed, dict):
        return {
            'features': empty_feature_dict(),
            'parse_success': False,
            'parse_error': 'Unable to parse JSON object from model response.',
        }

    normalized_input = {_normalize_key(key): value for key, value in parsed.items()}
    key_map = {
        'newsrelevance': 'News Relevance',
        'sentiment': 'Sentiment',
        'priceimpactpotential': 'Price Impact Potential',
        'trenddirection': 'Trend Direction',
        'earningsimpact': 'Earnings Impact',
        'investorconfidence': 'Investor Confidence',
        'riskprofilechange': 'Risk Profile Change',
    }

    normalized_output = empty_feature_dict()
    missing_keys = []
    conversion_errors = {}

    for normalized_key, output_key in key_map.items():
        if normalized_key not in normalized_input:
            missing_keys.append(output_key)
            continue

        value = normalized_input[normalized_key]
        try:
            normalized_output[output_key] = int(str(value).strip())
        except (ValueError, TypeError):
            conversion_errors[output_key] = value

    parse_error_parts = []
    if missing_keys:
        parse_error_parts.append(f"Missing keys: {', '.join(missing_keys)}")
    if conversion_errors:
        formatted_errors = ', '.join(f"{key}={value!r}" for key, value in conversion_errors.items())
        parse_error_parts.append(f"Non-integer values: {formatted_errors}")

    return {
        'features': normalized_output,
        'parse_success': len(parse_error_parts) == 0,
        'parse_error': '; '.join(parse_error_parts) if parse_error_parts else '',
    }

def run_inference(model, tokenizer, stock_data_path, ticker, model_label="vanilla", preview_first_response=True):
    df = pd.read_csv(stock_data_path)
    profile = COMPANY_PROFILES[ticker]
    results = []
    preview_printed = False

    for i in tqdm(range(len(df)), desc=f"{ticker} ({model_label})"):
        row = df.iloc[i]
        company_info = format_company_info(profile, row['Adj Close Price'], row['Bin Label'])
        news_str = format_news(row['News']) if pd.notna(row['News']) else "No recent news available."
        pr_str = format_press_releases(row['PressReleases']) if pd.notna(row['PressReleases']) else "No recent press releases available."
        input_text = INPUT_TEMPLATE.format(
            company_info=company_info,
            symbol=ticker,
            news=news_str,
            press_releases=pr_str,
        )
        prompt = ALPACA_PROMPT.format(SYSTEM_PROMPT, input_text, "")
        inputs = tokenizer(text=[prompt], return_tensors="pt")
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        prompt_length = inputs['input_ids'].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_tokens = outputs[:, prompt_length:]
        response = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
        parsed_response = parse_model_response(response)
        features = parsed_response['features']

        if preview_first_response and not preview_printed:
            print("Sample raw QLoRA response preview:")
            print(response[:1000])
            print("Parse success:", parsed_response['parse_success'])
            if parsed_response['parse_error']:
                print("Parse error:", parsed_response['parse_error'])
            print("Parsed features:", features)
            preview_printed = True

        results.append({
            'Date': row['Date'],
            'Adj Close Price': row['Adj Close Price'],
            'Returns': row['Returns'],
            'Bin Label': row['Bin Label'],
            'News Relevance': features['News Relevance'],
            'Sentiment': features['Sentiment'],
            'Price Impact Potential': features['Price Impact Potential'],
            'Trend Direction': features['Trend Direction'],
            'Earnings Impact': features['Earnings Impact'],
            'Investor Confidence': features['Investor Confidence'],
            'Risk Profile Change': features['Risk Profile Change'],
            'Parse Success': parsed_response['parse_success'],
            'Parse Error': parsed_response['parse_error'],
            'Prompt': input_text,
            'Raw Response': response,
        })

    results_df = pd.DataFrame(results)
    output_path = os.path.join(OUTPUT_DIR, f"qwen_{model_label}_ppo_{ticker}_data.csv")
    results_df.to_csv(output_path, index=False)
    print(f"Saved {len(results_df)} rows to {output_path}")

    nlp_cols = ['News Relevance', 'Sentiment', 'Price Impact Potential', 'Trend Direction', 'Earnings Impact', 'Investor Confidence', 'Risk Profile Change']
    numeric_features = results_df[nlp_cols].apply(pd.to_numeric, errors='coerce')
    parse_success_count = int(results_df['Parse Success'].fillna(False).sum())
    parse_failure_count = len(results_df) - parse_success_count
    all_zero = int(((numeric_features.fillna(0) == 0).all(axis=1) & results_df['Parse Success'].fillna(False)).sum())
    non_zero = int(((numeric_features.fillna(0) != 0).any(axis=1) & results_df['Parse Success'].fillna(False)).sum())
    print(f"  Parsed successfully: {parse_success_count}/{len(results_df)}")
    print(f"  Parse failures: {parse_failure_count}/{len(results_df)}")
    print(f"  Parsed non-zero predictions: {non_zero}/{len(results_df)} ({non_zero/len(results_df)*100:.1f}%)")
    print(f"  Parsed all-zero predictions: {all_zero}/{len(results_df)} ({all_zero/len(results_df)*100:.1f}%)")
    print(f"  Feature means (parsed rows): {numeric_features.mean(skipna=True).to_dict()}")
    return results_df

print("QLoRA inference helpers initialized in this cell.")
FastLanguageModel.for_inference(model_qlora)

qlora_results = {}
for ticker in TICKERS:
    path = stock_data_paths[ticker]
    if os.path.exists(path):
        print(f"\n{'='*60}")
        print(f"Running QLoRA inference on {ticker}...")
        print(f"{'='*60}")
        qlora_results[ticker] = run_inference(model_qlora, tokenizer, path, ticker, "qlora")
    else:
        print(f"Skipping {ticker}: data file not found")

if 'model_qlora' in globals():
    del model_qlora
if 'qlora_trainer' in globals():
    del qlora_trainer
torch.cuda.empty_cache()
print("\nQLoRA inference complete. Model unloaded from GPU.")

## Section 5: QA-LoRA Fine-Tuning

Fine-tune using **QA-LoRA** (Quantization-Aware LoRA).

**Key differences from QLoRA:**
- Uses PEFT's native `use_qalora=True` adapter implementation
- Adds `qalora_group_size` to make the adapter quantization-aware during fine-tuning
- Keeps the same rank and target modules so the comparison stays focused on the adapter method
- Avoids conflating QA-LoRA with rank-stabilized LoRA (`use_rslora`)

In [ ]:
# ============================================================
# Load model for QA-LoRA fine-tuning
# ============================================================

import os

os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

required_symbols = [
    "FastLanguageModel",
    "LoraConfig",
    "TaskType",
    "get_peft_model",
    "prepare_model_for_kbit_training",
]
missing_symbols = [name for name in required_symbols if name not in globals()]
if missing_symbols:
    raise RuntimeError(
        "Run the bootstrap cell near the top of the notebook first so Unsloth loads before trl / transformers / peft. "
        f"Missing symbols: {missing_symbols}"
    )

print("Loading model for QA-LoRA fine-tuning...")
model_qalora, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=TRAIN_MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
 )

# Prepare the quantized model for PEFT k-bit training
model_qalora = prepare_model_for_kbit_training(model_qalora)
if hasattr(model_qalora, "gradient_checkpointing_enable"):
    model_qalora.gradient_checkpointing_enable()
if hasattr(model_qalora, "enable_input_require_grads"):
    model_qalora.enable_input_require_grads()
model_qalora.config.use_cache = False

# Add QA-LoRA adapters
# QA-LoRA uses PEFT's native quantization-aware LoRA implementation via use_qalora=True
qalora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_rslora=False,
    use_qalora=True,
    qalora_group_size=QALORA_GROUP_SIZE,
)
model_qalora = get_peft_model(model_qalora, qalora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model_qalora.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_qalora.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")
print(f"QA-LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, use_qalora=True, group_size={QALORA_GROUP_SIZE}")
print(f"Training max sequence length: {TRAIN_MAX_SEQ_LENGTH}")
print(f"UNSLOTH_USE_MODELSCOPE={os.environ['UNSLOTH_USE_MODELSCOPE']}")
print(f"WANDB_DISABLED={os.environ['WANDB_DISABLED']}")

In [ ]:
# ============================================================
# Train QA-LoRA model
# ============================================================

if "SFTTrainer" not in globals() or "TrainingArguments" not in globals() or "is_bfloat16_supported" not in globals():
    raise RuntimeError(
        "Run the bootstrap cell near the top of the notebook first so the training libraries are initialized safely."
    )

if "dataset" not in globals() or dataset is None:
    print("Training dataset not found in memory; rebuilding the cleaned dataset for this runtime...")
    import ast
    import json
    from datasets import Dataset
    from transformers import AutoTokenizer

    if "training_data" not in globals() or training_data is None:
        if "training_data_path" not in globals():
            raise RuntimeError("Run the Google Drive/data loading cells first so training_data_path is defined.")
        with open(training_data_path) as file:
            training_data = json.load(file)

    if "ALPACA_PROMPT" not in globals():
        ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

    if "tokenizer" not in globals() or tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
        if tokenizer.pad_token is None and tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token

    if "SIGNAL_KEYS" not in globals():
        SIGNAL_KEYS = [
            "news_relevance",
            "sentiment",
            "price_impact_potential",
            "trend_direction",
            "earnings_impact",
            "investor_confidence",
            "risk_profile_change",
        ]

    if "parse_training_response" not in globals():
        def parse_training_response(raw_response):
            parsed = raw_response
            if isinstance(raw_response, str):
                try:
                    parsed = json.loads(raw_response)
                except json.JSONDecodeError:
                    try:
                        parsed = ast.literal_eval(raw_response)
                    except (ValueError, SyntaxError):
                        parsed = None

            if not isinstance(parsed, dict):
                return None, "response is not a JSON object"

            normalized = {}
            for key in SIGNAL_KEYS:
                if key not in parsed:
                    return None, f"missing key: {key}"
                value = parsed[key]
                if value is None:
                    return None, f"null value: {key}"
                try:
                    normalized[key] = int(str(value).strip())
                except (ValueError, TypeError):
                    return None, f"non-integer value for {key}: {value!r}"

            return normalized, ""

    if "build_clean_training_dataset" not in globals():
        def build_clean_training_dataset(records, tokenizer, alpaca_prompt, verbose=True):
            eos_token = tokenizer.eos_token or ""
            clean_rows = []
            invalid_reason_counts = {}
            invalid_examples = []
            zero_signal_count = 0

            for index, item in enumerate(records):
                instruction = item.get("instruction")
                input_text = item.get("input")
                if not isinstance(instruction, str) or not isinstance(input_text, str):
                    reason = "missing instruction/input text"
                    invalid_reason_counts[reason] = invalid_reason_counts.get(reason, 0) + 1
                    if len(invalid_examples) < 5:
                        invalid_examples.append({"index": index, "reason": reason})
                    continue

                normalized_response, reason = parse_training_response(item.get("response"))
                if normalized_response is None:
                    invalid_reason_counts[reason] = invalid_reason_counts.get(reason, 0) + 1
                    if len(invalid_examples) < 5:
                        invalid_examples.append({"index": index, "reason": reason})
                    continue

                zero_signal_count += int(all(normalized_response[key] == 0 for key in SIGNAL_KEYS))
                clean_rows.append({
                    "instruction": instruction,
                    "input": input_text,
                    "output": json.dumps(normalized_response, ensure_ascii=False),
                })

            if not clean_rows:
                raise ValueError("No valid training rows remained after filtering invalid labels.")

            dataset_dict = {
                "instruction": [row["instruction"] for row in clean_rows],
                "input": [row["input"] for row in clean_rows],
                "output": [row["output"] for row in clean_rows],
            }
            dataset = Dataset.from_dict(dataset_dict)

            def formatting_prompts_func(examples):
                texts = []
                for instruction, input_text, output in zip(examples["instruction"], examples["input"], examples["output"]):
                    texts.append(alpaca_prompt.format(instruction, input_text, output) + eos_token)
                return {"text": texts}

            dataset = dataset.map(formatting_prompts_func, batched=True)

            stats = {
                "raw_rows": len(records),
                "valid_rows": len(clean_rows),
                "dropped_rows": len(records) - len(clean_rows),
                "zero_signal_rows": zero_signal_count,
                "non_zero_signal_rows": len(clean_rows) - zero_signal_count,
                "invalid_reason_counts": invalid_reason_counts,
                "invalid_examples": invalid_examples,
            }

            if verbose:
                print(f"Loaded {stats['raw_rows']} raw training rows")
                print(f"Kept {stats['valid_rows']} valid rows and dropped {stats['dropped_rows']} invalid rows")
                print(f"All-zero label rows kept: {stats['zero_signal_rows']}")
                print(f"Non-zero label rows kept: {stats['non_zero_signal_rows']}")
                if invalid_reason_counts:
                    print("Dropped-row reasons:")
                    for reason, count in sorted(invalid_reason_counts.items(), key=lambda item: (-item[1], item[0])):
                        print(f"  - {reason}: {count}")
                if invalid_examples:
                    print("Sample dropped rows:")
                    for sample in invalid_examples:
                        print(f"  - index {sample['index']}: {sample['reason']}")

            return dataset, clean_rows, stats

    dataset, clean_training_records, training_dataset_stats = build_clean_training_dataset(
        training_data,
        tokenizer=tokenizer,
        alpaca_prompt=ALPACA_PROMPT,
        verbose=True,
    )
    print(f"Rebuilt cleaned training dataset with {len(dataset)} examples")

gpu_mem_before = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory before training: {gpu_mem_before:.2f} GB")
if gpu_mem_before > 60:
    print("WARNING: GPU memory is already extremely high before training.")
    print("Recommended safe starting point: BATCH_SIZE=1 and TRAIN_MAX_SEQ_LENGTH=1024 in Cell 3, then restart the runtime.")

print(f"Starting QA-LoRA training ({NUM_EPOCHS} epoch, {len(dataset)} samples)...")
print(f"Training config: batch_size={BATCH_SIZE}, grad_accum={GRADIENT_ACCUMULATION}, max_seq_length={TRAIN_MAX_SEQ_LENGTH}")
if "training_dataset_stats" in globals():
    print(f"Training rows kept after label validation: {training_dataset_stats['valid_rows']}")
    print(f"Training rows dropped after label validation: {training_dataset_stats['dropped_rows']}")

if hasattr(model_qalora, "config"):
    model_qalora.config.use_cache = False

torch.cuda.empty_cache()

qalora_trainer = SFTTrainer(
    model=model_qalora,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=TRAIN_MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        gradient_checkpointing=True,
        logging_steps=10,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir=os.path.join(OUTPUT_DIR, "qalora_checkpoints"),
        report_to="none",
    ),
)

qalora_stats = qalora_trainer.train()

print(f"\nTraining complete!")
print(f"  Training time: {qalora_stats.metrics['train_runtime']:.1f} seconds")
print(f"  Training loss: {qalora_stats.metrics['train_loss']:.4f}")
print(f"  GPU memory after training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ============================================================
# Save QA-LoRA model
# ============================================================

import shutil

qalora_save_path = os.path.join(OUTPUT_DIR, "qwen_qalora_adapter")
model_qalora.save_pretrained(qalora_save_path)
tokenizer.save_pretrained(qalora_save_path)
print(f"QA-LoRA adapter saved to {qalora_save_path}")

# Also try to save a merged model for easier inference, but keep the adapter as the canonical artifact
qalora_merged_path = os.path.join(OUTPUT_DIR, "qwen_qalora_merged")
if os.path.exists(qalora_merged_path):
    shutil.rmtree(qalora_merged_path, ignore_errors=True)

try:
    model_qalora.save_pretrained_merged(qalora_merged_path, tokenizer, save_method="merged_16bit")
    print(f"QA-LoRA merged model saved to {qalora_merged_path}")
except Exception as error:
    print(f"Merged QA-LoRA export failed: {type(error).__name__}: {error}")
    if os.path.exists(qalora_merged_path):
        shutil.rmtree(qalora_merged_path, ignore_errors=True)
        print("Removed incomplete merged QA-LoRA directory after failed export.")
    print("Continue with the saved adapter path; the QA-LoRA inference cell will reload the base model + adapter.")

In [ ]:
# ============================================================
# Run QA-LoRA Inference on All 5 Stocks
# ============================================================

import os
import json
import ast
import re
import glob
import shutil
import pandas as pd
import torch
from tqdm import tqdm

if "FastLanguageModel" not in globals():
    raise RuntimeError("Run the bootstrap cell first so `FastLanguageModel` is available.")

from peft import PeftModel

qalora_merged_path = os.path.join(OUTPUT_DIR, "qwen_qalora_merged")
qalora_adapter_path = os.path.join(OUTPUT_DIR, "qwen_qalora_adapter")

def has_valid_merged_checkpoint(model_dir):
    if not os.path.isdir(model_dir):
        return False
    if os.path.isfile(os.path.join(model_dir, "adapter_config.json")):
        return False
    if not os.path.isfile(os.path.join(model_dir, "config.json")):
        return False
    candidate_patterns = ["model*.safetensors", "*.safetensors", "*.bin"]
    for pattern in candidate_patterns:
        for file_path in glob.glob(os.path.join(model_dir, pattern)):
            file_name = os.path.basename(file_path).lower()
            if "adapter" in file_name:
                continue
            try:
                if os.path.getsize(file_path) > 0:
                    return True
            except OSError:
                continue
    return False

merged_available = has_valid_merged_checkpoint(qalora_merged_path)
adapter_available = os.path.isdir(qalora_adapter_path)

if not merged_available and os.path.isdir(qalora_merged_path):
    print(f"Skipping invalid merged QA-LoRA directory (missing full-model weights or adapter-only artifacts): {qalora_merged_path}")

if not merged_available and not adapter_available:
    raise RuntimeError(
        "No valid saved `qwen_qalora_merged` weights or `qwen_qalora_adapter` folder found in OUTPUT_DIR. "
        "Run the QA-LoRA save cell first so the model is written to Google Drive."
    )

model_qalora = None
last_load_error = None

if merged_available:
    try:
        print(f"Trying merged QA-LoRA model from Drive: {qalora_merged_path}")
        merged_dtype = torch.bfloat16 if ("is_bfloat16_supported" in globals() and is_bfloat16_supported()) else torch.float16
        model_qalora, tokenizer = FastLanguageModel.from_pretrained(
            model_name=qalora_merged_path,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=merged_dtype,
            load_in_4bit=False,
        )
        print("Loaded merged QA-LoRA model successfully.")
    except Exception as error:
        last_load_error = error
        print(f"Merged QA-LoRA load failed: {type(error).__name__}: {error}")
        if os.path.isdir(qalora_merged_path):
            shutil.rmtree(qalora_merged_path, ignore_errors=True)
            print("Removed corrupted merged QA-LoRA directory after failed load.")
        print("Falling back to base model + saved QA-LoRA adapter.")

if model_qalora is None:
    if not adapter_available:
        raise RuntimeError(
            "Merged QA-LoRA checkpoint could not be loaded, and `qwen_qalora_adapter` was not found in OUTPUT_DIR."
        ) from last_load_error

    print(f"Loading base model for adapter inference: {MODEL_NAME}")
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    print(f"Loading QA-LoRA adapter from Drive: {qalora_adapter_path}")
    model_qalora = PeftModel.from_pretrained(base_model, qalora_adapter_path)
    print("Loaded QA-LoRA adapter on top of the base model successfully.")

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

SYSTEM_PROMPT = """[SYSTEM PROMPT] You are a senior quantitative analyst specializing in stock market analysis. Your task is to analyze the provided company recent news and press releases to generate key features that could influence the stock's price movement in the next trading session. Focus on interpreting the given data to provide insights for algorithmic trading models."""

INPUT_TEMPLATE = """[COMPANY BASICS]
{company_info}

[RECENT NEWS]
Here are the recent news articles related to {symbol}:
{news}

[LATEST PRESS RELEASE]
Here is the most recent press release from {symbol} (if available):
{press_releases}

[ANALYSIS TASKS]
Based primarily on the provided news and press releases (if available) generate the following features defined in the output format. When analyzing, pay special attention to:
1. Potential negative impacts or risks, even if they're subtle or not the main focus of the news.
2. Market saturation signs, increased competition, or regulatory challenges.
3. Discrepancies between the tone of the news and the actual content.
4. Possible overoptimism in positive news that might lead to unrealistic expectations.
5. Short-term versus long-term implications of the news, especially potential short-term negative reactions.
Be cautious of overly positive sentiment and ensure you're giving appropriate weight to any negative indicators.

[FUTURE INFORMATION]
Future price information is unavailable at inference time. Generate the features using only the company basics, recent news, and press releases.

[OUTPUT FORMAT]
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing ```json and ```:
```json
{{
  "news_relevance": string,
  "sentiment": string,
  "price_impact_potential": string,
  "trend_direction": string,
  "earnings_impact": string,
  "investor_confidence": string,
  "risk_profile_change": string
}}
```
IMPORTANT: Your response should ONLY include the JSON structure or the markdown code snippet containing it. Do not include any additional explanation or analysis."""

def format_company_info(profile, adj_close, bin_label):
    price_change_desc = bin_label.replace('U', 'up by ').replace('D', 'down by ')
    price_change_desc = price_change_desc.replace('1', '0-1%').replace('2', '1-2%').replace('3', '2-3%').replace('4', '3-4%')
    if '5+' in price_change_desc:
        price_change_desc = price_change_desc.replace('5+', 'more than 5%')
    else:
        price_change_desc = price_change_desc.replace('5', '4-5%')
    return (
        f"{profile['name']} is a company trading under the ticker {profile['symbol']}. "
        f"The company operates in the {profile['industry']} industry with a market capitalization of "
        f"${profile['marketCapitalization']:,.0f}. "
        f"It has {profile['employeeTotal']:,} employees. "
        f"The current stock price is ${adj_close:.2f}, with today's price change {price_change_desc} compared to the previous closing price."
    )

def format_news(news_json_str):
    try:
        news = json.loads(news_json_str)
        if not news:
            return "No recent news available."
        sampled = news[:10]
        return "\n".join([f"[Headline]: {item['headline']}\n[Summary]: {item['summary']}" for item in sampled])
    except Exception:
        return "No recent news available."

def format_press_releases(pr_json_str):
    try:
        press_releases = json.loads(pr_json_str)
        if not press_releases:
            return "No recent press releases available."
        return "\n".join([f"[Headline]: {item['headline']}\n[Description]: {item['description']}" for item in press_releases[:5]])
    except Exception:
        return "No recent press releases available."

def _normalize_key(key):
    return re.sub(r'[^a-z0-9]+', '', str(key).strip().lower())

def empty_feature_dict(fill_value=pd.NA):
    return {
        'News Relevance': fill_value,
        'Sentiment': fill_value,
        'Price Impact Potential': fill_value,
        'Trend Direction': fill_value,
        'Earnings Impact': fill_value,
        'Investor Confidence': fill_value,
        'Risk Profile Change': fill_value,
    }

def parse_model_response(response_text):
    cleaned = response_text.strip()
    response_match = re.search(r'### Response:\s*(.*)', cleaned, re.DOTALL)
    if response_match:
        cleaned = response_match.group(1).strip()

    cleaned = re.sub(r'^```json\s*', '', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'^```\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)

    json_match = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if not json_match:
        return {
            'features': empty_feature_dict(),
            'parse_success': False,
            'parse_error': 'No JSON object found in model response.',
        }

    candidate = json_match.group(0)
    parsed = None
    try:
        parsed = json.loads(candidate)
    except json.JSONDecodeError:
        try:
            parsed = ast.literal_eval(candidate)
        except (ValueError, SyntaxError):
            parsed = None

    if not isinstance(parsed, dict):
        return {
            'features': empty_feature_dict(),
            'parse_success': False,
            'parse_error': 'Unable to parse JSON object from model response.',
        }

    normalized_input = {_normalize_key(key): value for key, value in parsed.items()}
    key_map = {
        'newsrelevance': 'News Relevance',
        'sentiment': 'Sentiment',
        'priceimpactpotential': 'Price Impact Potential',
        'trenddirection': 'Trend Direction',
        'earningsimpact': 'Earnings Impact',
        'investorconfidence': 'Investor Confidence',
        'riskprofilechange': 'Risk Profile Change',
    }

    normalized_output = empty_feature_dict()
    missing_keys = []
    conversion_errors = {}

    for normalized_key, output_key in key_map.items():
        if normalized_key not in normalized_input:
            missing_keys.append(output_key)
            continue

        value = normalized_input[normalized_key]
        try:
            normalized_output[output_key] = int(str(value).strip())
        except (ValueError, TypeError):
            conversion_errors[output_key] = value

    parse_error_parts = []
    if missing_keys:
        parse_error_parts.append(f"Missing keys: {', '.join(missing_keys)}")
    if conversion_errors:
        formatted_errors = ', '.join(f"{key}={value!r}" for key, value in conversion_errors.items())
        parse_error_parts.append(f"Non-integer values: {formatted_errors}")

    return {
        'features': normalized_output,
        'parse_success': len(parse_error_parts) == 0,
        'parse_error': '; '.join(parse_error_parts) if parse_error_parts else '',
    }

def run_inference(model, tokenizer, stock_data_path, ticker, model_label="vanilla", preview_first_response=True):
    df = pd.read_csv(stock_data_path)
    profile = COMPANY_PROFILES[ticker]
    results = []
    preview_printed = False

    for i in tqdm(range(len(df)), desc=f"{ticker} ({model_label})"):
        row = df.iloc[i]
        company_info = format_company_info(profile, row['Adj Close Price'], row['Bin Label'])
        news_str = format_news(row['News']) if pd.notna(row['News']) else "No recent news available."
        pr_str = format_press_releases(row['PressReleases']) if pd.notna(row['PressReleases']) else "No recent press releases available."
        input_text = INPUT_TEMPLATE.format(
            company_info=company_info,
            symbol=ticker,
            news=news_str,
            press_releases=pr_str,
        )
        prompt = ALPACA_PROMPT.format(SYSTEM_PROMPT, input_text, "")
        inputs = tokenizer(text=[prompt], return_tensors="pt")
        inputs = {key: value.to(model.device) for key, value in inputs.items()}
        prompt_length = inputs['input_ids'].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_tokens = outputs[:, prompt_length:]
        response = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
        parsed_response = parse_model_response(response)
        features = parsed_response['features']

        if preview_first_response and not preview_printed:
            print("Sample raw QA-LoRA response preview:")
            print(response[:1000])
            print("Parse success:", parsed_response['parse_success'])
            if parsed_response['parse_error']:
                print("Parse error:", parsed_response['parse_error'])
            print("Parsed features:", features)
            preview_printed = True

        results.append({
            'Date': row['Date'],
            'Adj Close Price': row['Adj Close Price'],
            'Returns': row['Returns'],
            'Bin Label': row['Bin Label'],
            'News Relevance': features['News Relevance'],
            'Sentiment': features['Sentiment'],
            'Price Impact Potential': features['Price Impact Potential'],
            'Trend Direction': features['Trend Direction'],
            'Earnings Impact': features['Earnings Impact'],
            'Investor Confidence': features['Investor Confidence'],
            'Risk Profile Change': features['Risk Profile Change'],
            'Parse Success': parsed_response['parse_success'],
            'Parse Error': parsed_response['parse_error'],
            'Prompt': input_text,
            'Raw Response': response,
        })

    results_df = pd.DataFrame(results)
    output_path = os.path.join(OUTPUT_DIR, f"qwen_{model_label}_ppo_{ticker}_data.csv")
    results_df.to_csv(output_path, index=False)
    print(f"Saved {len(results_df)} rows to {output_path}")

    nlp_cols = ['News Relevance', 'Sentiment', 'Price Impact Potential', 'Trend Direction', 'Earnings Impact', 'Investor Confidence', 'Risk Profile Change']
    numeric_features = results_df[nlp_cols].apply(pd.to_numeric, errors='coerce')
    parse_success_count = int(results_df['Parse Success'].fillna(False).sum())
    parse_failure_count = len(results_df) - parse_success_count
    all_zero = int(((numeric_features.fillna(0) == 0).all(axis=1) & results_df['Parse Success'].fillna(False)).sum())
    non_zero = int(((numeric_features.fillna(0) != 0).any(axis=1) & results_df['Parse Success'].fillna(False)).sum())
    print(f"  Parsed successfully: {parse_success_count}/{len(results_df)}")
    print(f"  Parse failures: {parse_failure_count}/{len(results_df)}")
    print(f"  Parsed non-zero predictions: {non_zero}/{len(results_df)} ({non_zero/len(results_df)*100:.1f}%)")
    print(f"  Parsed all-zero predictions: {all_zero}/{len(results_df)} ({all_zero/len(results_df)*100:.1f}%)")
    print(f"  Feature means (parsed rows): {numeric_features.mean(skipna=True).to_dict()}")
    return results_df

print("QA-LoRA inference helpers initialized in this cell.")
FastLanguageModel.for_inference(model_qalora)

qalora_results = {}
for ticker in TICKERS:
    path = stock_data_paths[ticker]
    if os.path.exists(path):
        print(f"\n{'='*60}")
        print(f"Running QA-LoRA inference on {ticker}...")
        print(f"{'='*60}")
        qalora_results[ticker] = run_inference(model_qalora, tokenizer, path, ticker, "qalora")
    else:
        print(f"Skipping {ticker}: data file not found")

if 'model_qalora' in globals():
    del model_qalora
if 'qalora_trainer' in globals():
    del qalora_trainer
torch.cuda.empty_cache()
print("\nQA-LoRA inference complete. Model unloaded from GPU.")

## Section 6: Combine Results & Save for Download

Combine per-stock results into combined CSV files (matching the format used by the RL training pipeline).

In [ ]:
# ============================================================
# Combine per-stock results and save
# ============================================================

import os
import pandas as pd

nlp_cols = [
    'News Relevance', 'Sentiment', 'Price Impact Potential',
    'Trend Direction', 'Earnings Impact', 'Investor Confidence', 'Risk Profile Change'
]

MODEL_OUTPUT_LABELS = {
    'base': 'vanilla_results',
    'qlora': 'qlora_results',
    'qalora': 'qalora_results',
}

def load_saved_results(model_label):
    loaded = {}
    for ticker in TICKERS:
        stock_path = os.path.join(OUTPUT_DIR, f"qwen_{model_label}_ppo_{ticker}_data.csv")
        if os.path.exists(stock_path):
            loaded[ticker] = pd.read_csv(stock_path)
    return loaded

for model_label, global_name in MODEL_OUTPUT_LABELS.items():
    results_dict = globals().get(global_name)
    if not isinstance(results_dict, dict) or not results_dict:
        results_dict = load_saved_results(model_label)
        globals()[global_name] = results_dict
        print(f"Reconstructed `{global_name}` from saved CSVs: {list(results_dict.keys())}")

    combined_dfs = []
    for ticker in TICKERS:
        if ticker in results_dict:
            df = results_dict[ticker].copy()
            df['ticker'] = ticker
            combined_dfs.append(df)

            stock_path = os.path.join(OUTPUT_DIR, f"qwen_{model_label}_ppo_{ticker}_data.csv")
            df.to_csv(stock_path, index=False)
            print(f"{model_label} {ticker}: saved {len(df)} rows")

    if combined_dfs:
        combined = pd.concat(combined_dfs, ignore_index=True)
        combined_path = os.path.join(OUTPUT_DIR, f"qwen_{model_label}_ppo_data.csv")
        combined.to_csv(combined_path, index=False)

        all_zero = (combined[nlp_cols] == 0).all(axis=1).sum()
        non_zero = len(combined) - all_zero
        print(f"  Combined: {len(combined)} rows, {non_zero} non-zero ({non_zero/len(combined)*100:.1f}%)")
        print("  Feature means:")
        for col in nlp_cols:
            print(f"    {col}: {combined[col].mean():.3f} (std: {combined[col].std():.3f})")
        print()
    else:
        print(f"No saved or in-memory results found for `{model_label}` in {OUTPUT_DIR}")

In [ ]:
# ============================================================
# Compare Vanilla vs QLoRA vs QA-LoRA
# ============================================================

import os
import pandas as pd

if 'nlp_cols' not in globals():
    nlp_cols = [
        'News Relevance', 'Sentiment', 'Price Impact Potential',
        'Trend Direction', 'Earnings Impact', 'Investor Confidence', 'Risk Profile Change'
    ]

MODEL_COMPARE_SPECS = [
    ('Vanilla', 'base', 'vanilla_results'),
    ('QLoRA', 'qlora', 'qlora_results'),
    ('QA-LoRA', 'qalora', 'qalora_results'),
]

def load_saved_results(model_label):
    loaded = {}
    for ticker in TICKERS:
        stock_path = os.path.join(OUTPUT_DIR, f"qwen_{model_label}_ppo_{ticker}_data.csv")
        if os.path.exists(stock_path):
            loaded[ticker] = pd.read_csv(stock_path)
    return loaded

for display_label, model_label, global_name in MODEL_COMPARE_SPECS:
    results_dict = globals().get(global_name)
    if not isinstance(results_dict, dict) or not results_dict:
        results_dict = load_saved_results(model_label)
        globals()[global_name] = results_dict
        print(f"Loaded `{global_name}` for comparison from saved CSVs: {list(results_dict.keys())}")

print("=" * 70)
print("COMPARISON: Vanilla Qwen vs QLoRA vs QA-LoRA")
print("=" * 70)

for ticker in TICKERS:
    print(f"\n--- {ticker} ---")
    for display_label, model_label, global_name in MODEL_COMPARE_SPECS:
        results_dict = globals().get(global_name, {})
        if ticker in results_dict:
            df = results_dict[ticker]
            all_zero = (df[nlp_cols] == 0).all(axis=1).sum()
            non_zero = len(df) - all_zero
            print(
                f"  {display_label:8s}: {non_zero:3d}/{len(df)} non-zero ({non_zero/len(df)*100:.1f}%) | "
                f"Sentiment={df['Sentiment'].mean():+.3f} | "
                f"Trend={df['Trend Direction'].mean():+.3f} | "
                f"Impact={df['Price Impact Potential'].mean():+.3f}"
            )
        else:
            print(f"  {display_label:8s}: N/A")

print("\n" + "=" * 70)
print("All results saved to:", OUTPUT_DIR)
print("Files to download:")
print("  - qwen_base_ppo_data.csv (combined vanilla)")
print("  - qwen_qlora_ppo_data.csv (combined QLoRA)")
print("  - qwen_qalora_ppo_data.csv (combined QA-LoRA)")
print("  - qwen_base_ppo_{TICKER}_data.csv (per-stock vanilla)")
print("  - qwen_qlora_ppo_{TICKER}_data.csv (per-stock QLoRA)")
print("  - qwen_qalora_ppo_{TICKER}_data.csv (per-stock QA-LoRA)")
print("  - qwen_qlora_adapter/ (QLoRA weights)")
print("  - qwen_qalora_adapter/ (QA-LoRA weights)")
print("  - qwen_qlora_merged/ (merged model)")
print("  - qwen_qalora_merged/ (merged model)")

In [ ]:
# ============================================================
# Zip everything for easy download
# ============================================================

import shutil

# Create a zip file of all results
zip_path = os.path.join(OUTPUT_DIR, "qwen_llm_results")
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"Results zipped to: {zip_path}.zip")
print(f"File size: {os.path.getsize(zip_path + '.zip') / 1e6:.1f} MB")
print()
print("To download from Colab, run:")
print("  from google.colab import files")
print(f"  files.download('{zip_path}.zip')")
print()
print("Or copy from Google Drive:")
print(f"  {OUTPUT_DIR}")

## Section 7: Notes for Local Use

### Copying Results to Local Machine

After running this notebook on Colab, download the zip file and extract the CSV files into:
```
our_code/ppo_data/
  qwen_base_ppo_AAPL_data.csv
  qwen_base_ppo_AMZN_data.csv
  qwen_base_ppo_CRM_data.csv
  qwen_base_ppo_MSFT_data.csv
  qwen_base_ppo_NFLX_data.csv
  qwen_base_ppo_data.csv        (combined)
  qwen_qlora_ppo_AAPL_data.csv
  ... (per-stock and combined)
  qwen_qalora_ppo_data.csv      (combined)
```

These files will have the same format as `gemini_ppo_AAPL_data.csv` and can be directly used by the PPO/SAC training pipeline.

### Memory Requirements Recap

| Model | 4-bit Inference | QLoRA Training | Recommended GPU |
|-------|-----------------|----------------|-----------------|
| Qwen3.5-27B | ~15 GB | ~25 GB | Colab A100 (paid) |
| Qwen3.5-9B | ~6 GB | ~12 GB | Colab T4 (free) |

Your Mac (M4 Max, 36GB): Cannot efficiently fine-tune 27B model. Use Colab with A100 GPU runtime.

### QLoRA vs QA-LoRA Differences

| Aspect | QLoRA | QA-LoRA |
|--------|-------|---------|
| LoRA rank (r) | 16 | 16 |
| LoRA alpha | 16 | 16 |
| use_qalora | False | True |
| qalora_group_size | N/A | 16 |
| Quantization awareness | Frozen quantized base + standard LoRA | Group-wise quantization-aware LoRA adaptation |
| Goal | Efficient 4-bit adapter training | Better balance between quantization and adaptation |

The key difference is that QA-LoRA introduces quantization-aware group-wise adapter behavior through `use_qalora=True` and `qalora_group_size`, rather than using rank stabilization (`use_rslora`).